# Day 3 v2 — 関与率×ラグの感度分析（自己資金化トレードポートフォリオ方式）

**前回からの変更点（頂いた実データの結果を見て）**

前回のノートを実データで動かすと `participation` の年率122.7%・vol 11785%・maxdd -243% という
明らかに壊れた数値が出ました。原因はほぼ確実に **`daily_return` に -100%を下回る/極端に大きい
値が混入していて、ウェイトの日次ドリフト（`(1+r)`で割って再正規化する処理）がそこで発散した**こと
です。939銘柄×4054日という規模なら、コーポレートアクション未調整や欠損等による1件の異常値でも
十分に起こります。

今回は2つの対応をしています。
1. **データの健全性チェックと軽いクリップ**を Load 直後に追加（§1）。
2. **エンジンを「全体NAVを日々再正規化する」方式から、`(1+r)`で割る処理が一切不要な
   「各月の自己資金化トレードポートフォリオ（買い−売り、合計ゼロ）を作り、その日次リターンを
   足し合わせる」方式に変更**（§6）。個別銘柄の異常リターンが混入しても、影響はその銘柄の
   ウェイト分だけに限定され、発散しません。

**ご指示に基づくその他の変更**
- `lag2` / `mega`（固定カレンダースケジュール）は削除。比較対象は `benchmark` のみ。
- AUM=500億円を起点に、**関与率とラグを独立変数として** `PARTICIPATION_GRID` × `LAG_GRID`
  でスイープ（§8）。
- 銘柄の流動性（売買高ベース）で分位分けし、流動性の低い銘柄群だけに別処理を試す比較を追加（§9）。
- 片道ターンオーバーを計測（§7）。
- 各リバランス月で自己資金化トレードポートフォリオを組成し、その累積リターンを「正規の」
  パフォーマンス指標として追跡（§6・§8）。
- 最後に取引コスト導入の考え方だけメモ（§10、実装はまだ先）。

⚠️ 実データはこのセッションに無いため、Day2と同じ列名・スキーマの合成データ（かつ意図的に
異常リターンを1件混入させたもの）で最後まで動作確認しています。


## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 150)

DATA_DIR = Path("/home/ubuntu/work/data")
OUT_DIR = Path("/home/ubuntu/work/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

UNIVERSE_FILE = DATA_DIR / "jp_top500_universe_201001_202607.parquet"
RETURNS_FILE = DATA_DIR / "daily_returns_20100104_20260729.parquet"
PORTFOLIO_FILE = DATA_DIR / "jp_top500_monthly_bpr_portfolios_201001_202606.csv"

COLS = dict(
    ret_date="base_ymd", ret_code="stock_code", ret_value="daily_return",
    ret_price="price_close", ret_volume="volume",
    uni_date="date_eom", uni_code="code", uni_cap="mkt_cap",
    prt_date="DATE_EOM", prt_code="CODE", prt_weight="WEIGHT",
)
FREQ = 12
OI = ["#0072B2", "#009E73", "#E69F00", "#D55E00", "#56B4E9", "#CC79A7", "#F0E442", "#000000"]

# ---- change these freely; everything below reads from here ----
AUM_TARGET = 50_000_000_000                                    # 500億円 -- change manually as needed
PARTICIPATION_GRID = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20]       # 1% .. 20%
LAG_GRID = [0, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35, 40]          # business days
MAX_EXEC_DAYS = 260   # ~1 trading year cap on how long any single month's execution can run;
                       # raise if the low-participation-rate rows show a large share_unresolved


## 1. Load（Day2から引き継ぎ + データ健全性チェック）

In [ ]:
def read(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)

def tidy(path: Path, keys: tuple, name: str, dedupe: bool = False) -> pd.DataFrame:
    date_col, code_col, val_col = keys
    df = read(path)
    missing = [c for c in keys if c not in df.columns]
    assert not missing, f"{path.name}: missing {missing}. Present: {list(df.columns)}"
    out = pd.DataFrame({
        "date": pd.to_datetime(df[date_col].astype(str)) if str(df[date_col].dtype).startswith(("int", "Int")) else pd.to_datetime(df[date_col]),
        "code": df[code_col].astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.upper(),
        name: pd.to_numeric(df[val_col], errors="coerce"),
    }).dropna()
    return out.groupby(["date", "code"], as_index=False)[name].sum() if dedupe else out

def load_weights(path: Path, keys: tuple) -> pd.DataFrame:
    return tidy(path, keys, "weight", dedupe=True)

RET = tidy(RETURNS_FILE, (COLS["ret_date"], COLS["ret_code"], COLS["ret_value"]), "ret")
UNI = tidy(UNIVERSE_FILE, (COLS["uni_date"], COLS["uni_code"], COLS["uni_cap"]), "cap")
PRT = load_weights(PORTFOLIO_FILE, (COLS["prt_date"], COLS["prt_code"], COLS["prt_weight"]))

CAL = pd.DatetimeIndex(np.sort(RET["date"].unique()))
BENCH = UNI.assign(weight=UNI.groupby("date")["cap"].transform(lambda s: s / s.sum()))[["date", "code", "weight"]]
held = set(PRT["code"]) | set(BENCH["code"])
RET_HELD = RET.loc[RET["code"].isin(held), ["date", "code", "ret"]].copy()

# ---- NEW: sanity check -- a return can never be <= -100%; flag and clip anything insane ----
print("=== return sanity check (pre-clean) ===")
print(f"min ret: {RET_HELD['ret'].min():.4f}   max ret: {RET_HELD['ret'].max():.4f}")
n_impossible = (RET_HELD["ret"] <= -1.0).sum()
n_extreme = (RET_HELD["ret"].abs() > 1.0).sum()
print(f"rows with ret <= -100% (mathematically impossible -> definitely bad data): {n_impossible}")
print(f"rows with |ret| > 100%: {n_extreme}")
if n_impossible:
    print(RET_HELD.loc[RET_HELD["ret"] <= -1.0].sort_values("ret").head(10).to_string(index=False))
RET_CLIP_LO, RET_CLIP_HI = -0.99, 3.0   # generous on the upside (small caps do jump); hard floor on the downside
n_clip = ((RET_HELD["ret"] < RET_CLIP_LO) | (RET_HELD["ret"] > RET_CLIP_HI)).sum()
RET_HELD["ret"] = RET_HELD["ret"].clip(RET_CLIP_LO, RET_CLIP_HI)
print(f"clipped {n_clip} rows to [{RET_CLIP_LO}, {RET_CLIP_HI}] before any further use\n")

print(f"calendar : {len(CAL)} trading days, {CAL.min().date()} -> {CAL.max().date()}")
print(f"portfolio: {PRT['date'].nunique()} month-ends x {PRT.groupby('date').size().median():.0f} names")

def tidy_yen_volume(path: Path, date_col: str, code_col: str, price_col: str, vol_col: str) -> pd.DataFrame:
    df = read(path)
    out = pd.DataFrame({
        "date": pd.to_datetime(df[date_col].astype(str)) if str(df[date_col].dtype).startswith(("int", "Int")) else pd.to_datetime(df[date_col]),
        "code": df[code_col].astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.upper(),
        "yen_volume": pd.to_numeric(df[price_col], errors="coerce") * pd.to_numeric(df[vol_col], errors="coerce"),
    }).dropna()
    return out.groupby(["date", "code"], as_index=False)["yen_volume"].sum()

VOL = tidy_yen_volume(RETURNS_FILE, COLS["ret_date"], COLS["ret_code"], COLS["ret_price"], COLS["ret_volume"])
print(f"volume   : {VOL['date'].nunique()} trading days x {VOL['code'].nunique()} names, "
      f"median daily yen volume {VOL['yen_volume'].median():,.0f}")


## 2. Engine（Day2から引き継ぎ — ベンチマーク計算にのみ使用）

In [ ]:
def snap(dates) -> pd.DatetimeIndex:
    d = pd.DatetimeIndex(pd.to_datetime(dates))
    pos = CAL.get_indexer(d, method="ffill")
    assert (pos >= 0).all()
    return CAL[pos]

def shift_bd(dates, n: int) -> pd.DatetimeIndex:
    pos = CAL.get_indexer(snap(dates)) + n
    out = np.full(len(pos), np.datetime64("NaT"), dtype="datetime64[ns]")
    ok = (pos >= 0) & (pos < len(CAL))
    out[ok] = CAL.values[pos[ok]]
    return pd.DatetimeIndex(out)

def period_grid(form_dates, lag: int) -> pd.DataFrame:
    f = snap(np.sort(pd.unique(pd.to_datetime(form_dates))))
    e = shift_bd(f, lag)
    keep = e.notna()
    f, e = f[keep], e[keep]
    P = pd.DataFrame({"form": f[:-1], "start": e[:-1], "end": e[1:]})
    P["period"] = pd.PeriodIndex(pd.DatetimeIndex(P["form"]), freq="M") + 1
    return P

def stats(r: pd.Series, geometric: bool = True) -> pd.Series:
    """geometric=True for a level series (e.g. the benchmark). geometric=False for a
    DIFFERENCE/spread series (e.g. the trade-portfolio return below) -- Day2's own note
    applies: compounding a near-zero-mean difference series just measures variance drag,
    not a real cost, so those get the arithmetic (mean x FREQ) treatment instead."""
    r = r.dropna()
    if geometric:
        cum = (1 + r).cumprod()
        ann = (cum.iloc[-1] ** (FREQ / len(r)) - 1) * 100
        maxdd = (cum / cum.cummax() - 1).min() * 100
    else:
        ann = r.mean() * FREQ * 100
        cum = r.cumsum()
        maxdd = (cum - cum.cummax()).min() * 100
    vol = r.std(ddof=1) * np.sqrt(FREQ) * 100
    return pd.Series({
        "months": len(r), "ann_%": ann, "vol_%": vol, "maxdd_%": maxdd,
        "hit": (r > 0).mean(), "t": r.mean() / (r.std(ddof=1) / np.sqrt(len(r))),
        "ratio": ann / vol,
    })

def backtest(weights: pd.DataFrame, lag: int = 0, grid: pd.DataFrame | None = None, min_coverage: float = 0.5) -> pd.Series:
    """Instant, full-execution backtest -- used only for the benchmark now."""
    P = period_grid(weights["date"], lag) if grid is None else grid.reset_index(drop=True)
    d = RET_HELD["date"].values
    k = np.searchsorted(P["start"].values, d, side="left") - 1
    started = k >= 0
    ends = P["end"].values[np.where(started, k, 0)]
    inside = started & (d <= ends)
    rr = RET_HELD.loc[inside].assign(k=k[inside])
    days = rr.groupby("k")["date"].nunique()
    agg = (rr.assign(gr=1.0 + rr["ret"]).groupby(["k", "code"], sort=False)
           .agg(gross=("gr", "prod"), obs=("gr", "size")).reset_index())
    kmap = pd.Series(P.index, index=snap(P["form"]))
    w = weights.assign(k=snap(weights["date"]).map(kmap)).dropna(subset=["k"])
    w["k"] = w["k"].astype(int)
    j = w.merge(agg, on=["k", "code"], how="left")
    j["cov"] = j["obs"] / j["k"].map(days)
    ok = j["cov"].fillna(0.0) >= min_coverage
    j = j.loc[ok].copy()
    j["weight"] = j["weight"] / j.groupby("k")["weight"].transform("sum")
    j["contrib"] = j["weight"] * (j["gross"] - 1.0)
    out = P.copy()
    out["ret"] = out.index.map(j.groupby("k")["contrib"].sum())
    return out.set_index("period")["ret"]


## 3. Simulator setup（Day2から引き継ぎ）

`simulate_fund`（固定カレンダースケジュール）は今回は使いません — ご指示通り比較対象から外します。

In [ ]:
RW = RET_HELD.pivot_table(index="date", columns="code", values="ret", aggfunc="first").reindex(CAL)
CODES = RW.columns.to_numpy()
RMAT = np.nan_to_num(RW.to_numpy(dtype=float), nan=0.0)
CPOS = pd.Series(np.arange(len(CODES)), index=CODES)
DPOS = pd.Series(np.arange(len(CAL)), index=CAL)

def weight_vectors(weights: pd.DataFrame) -> dict:
    out = {}
    for d, g in weights.groupby("date"):
        v = np.zeros(len(CODES))
        j = CPOS.reindex(g["code"]).to_numpy(dtype=float)
        ok = ~np.isnan(j)
        v[j[ok].astype(int)] = g["weight"].to_numpy()[ok]
        s = v.sum()
        if s > 0:
            v /= s
        out[snap([d])[0]] = v
    return out

VOLW = VOL.pivot_table(index="date", columns="code", values="yen_volume", aggfunc="first").reindex(CAL).reindex(columns=CODES)
VOLMAT = np.nan_to_num(VOLW.to_numpy(dtype=float), nan=0.0)

WVEC = weight_vectors(PRT)
print(f"daily matrix: {RMAT.shape[0]} days x {RMAT.shape[1]} names; {len(WVEC)} formation dates")
print(f"VOLMAT ready: {VOLMAT.shape}, matches RMAT: {VOLMAT.shape == RMAT.shape}")


## 4. ベンチマーク（唯一の比較対象）

In [ ]:
_grid0 = period_grid(PRT["date"], 0)
bench_ret = backtest(BENCH, 0, grid=_grid0)
print(stats(bench_ret, True).round(3))


## 5. 流動性バケット（Ohtaの分位分けと同じ考え方）

サンプル全期間の平均日次売買高（ADV）で5分位に分けます。§9で「流動性の低い銘柄だけ別処理を
試す」ときに使います。


In [ ]:
ADV = pd.Series(np.where(VOLMAT.mean(axis=0) > 0, VOLMAT.mean(axis=0), np.nan), index=CODES, name="adv_yen")
LIQ_BUCKET = pd.qcut(ADV.rank(method="first"), 5, labels=["Q1 illiquid", "Q2", "Q3", "Q4", "Q5 liquid"])
print("median ADV by liquidity bucket (yen):")
print(ADV.groupby(LIQ_BUCKET).median().round(0))
ILLIQUID_CODES = set(LIQ_BUCKET[LIQ_BUCKET == "Q1 illiquid"].index)
ILLIQUID_POS = np.array([i for i, c in enumerate(CODES) if c in ILLIQUID_CODES])
print(f"\n{len(ILLIQUID_POS)} names in the illiquid (Q1) bucket")


## 6. 執行エンジン：各月の自己資金化トレードポートフォリオ

各リバランス月 $m$ で、$\Delta_m = w^{new}_m - w^{old}_m$（合計ゼロ、買い−売りの自己資金化
ポートフォリオ）を作ります。`lag` 営業日後から、その日の実売買高に基づく関与率キャップで
毎日埋めていきます（**完了しなくても次の月の決定日で強制終了しません** — 前回の「1ヶ月で
捌けない分は翌月の差分に混ざる」設計はやめて、必要な日数だけそのまま執行を続けます。重複する月の
執行はその日の寄与を単純に足し合わせます）。

**リターン規約**（指示通り、かつ全体NAVの再正規化が一切不要）：

$$r_{trade,t} = \sum_{m:\text{live}} \sum_i \left(\frac{q_{m,i,t-1}+q_{m,i,t}}{2}\right) r_{i,t}$$

$q_{m,i,t}$ は月 $m$ のトレードポートフォリオにおける銘柄 $i$ の累積執行量（ウェイト単位、0から
$\Delta_{m,i}$ に向かって単調に動く）。**`(1+r)`で割る処理がどこにも無いため、個別銘柄の異常
リターンが混入しても発散しません**（その銘柄のウェイト分だけの、有限で小さい影響に留まります）。

**累積のしかた**：Day2 の `stats()` 自身のコメント通り、差分（スプレッド）系列を幾何的に複利で
積むのは意味がなく（ほぼゼロ平均の系列を複利計算すると分散ドラッグ $-\sigma^2/2$ が「コスト」に
見えてしまう）、ここでも**算術（単純合計）**で累積します。


In [ ]:
def build_execution_path(delta_yen, vol_window, participation_cap, aum, max_days=None):
    """delta_yen: (n_names,) signed yen amount still to trade. vol_window: (n_days, n_names)
    real market yen volume. Returns q: (n_days, n_names) cumulative executed position, in
    WEIGHT units, monotonically moving from 0 toward delta_yen/aum."""
    cap_frac = participation_cap / (1.0 - participation_cap)
    n_days = len(vol_window) if max_days is None else min(max_days, len(vol_window))
    n = len(delta_yen)
    q = np.zeros((n_days, n))
    remaining = delta_yen.astype(float).copy()
    prev = np.zeros(n)
    for t in range(n_days):
        vol_today = vol_window[t]
        cap_i = np.where(vol_today > 0, cap_frac * vol_today, 0.0)
        trade = np.sign(remaining) * np.minimum(np.abs(remaining), cap_i)
        remaining = remaining - trade
        prev = prev + trade / aum
        q[t] = prev
    return q


def simulate_trade_portfolios(wvec: dict, aum: float = AUM_TARGET, participation_cap: float = 0.20,
                               lag: int = 2, max_exec_days: int = MAX_EXEC_DAYS):
    """Runs every month's self-financed trade portfolio independently (no forced monthly
    cutoff; overlapping months just add). Returns the aggregate daily r_trade series and a
    per-name-month days_to_complete diagnostic."""
    forms = pd.DatetimeIndex(sorted(wvec.keys()))
    n = len(CODES)
    r_trade = np.zeros(len(CAL))
    dtc_rows = []
    for m in range(1, len(forms)):
        d0, d1 = forms[m - 1], forms[m]
        delta_w = wvec[d1] - wvec[d0]
        active = delta_w != 0
        if not active.any():
            continue
        delta_yen = delta_w * aum
        start_date = shift_bd([d0], lag)[0]
        if pd.isna(start_date):
            continue
        i_start = int(DPOS[start_date])
        i_end = min(i_start + max_exec_days, len(CAL))
        if i_end <= i_start:
            continue
        vol_window = VOLMAT[i_start:i_end]
        q = build_execution_path(delta_yen, vol_window, participation_cap, aum)

        target_col = delta_w
        reached = np.abs(q - target_col[None, :]) <= 1.0 / aum
        for j in np.where(active)[0]:
            d2c = int(np.argmax(reached[:, j])) + 1 if reached[:, j].any() else np.nan
            dtc_rows.append({"decision_date": d0, "code": CODES[j],
                              "days_to_complete": d2c, "delta_yen": delta_yen[j]})

        r_mat_window = RMAT[i_start:i_end]
        q_before = np.vstack([np.zeros((1, n)), q[:-1]])
        eff = 0.5 * (q_before + q)
        r_contrib = np.einsum("tn,tn->t", eff, r_mat_window)
        r_trade[i_start:i_end] += r_contrib
    return pd.Series(r_trade, index=CAL, name="r_trade"), pd.DataFrame(dtc_rows)


def one_way_turnover(wvec: dict) -> pd.Series:
    """0.5 * sum |Delta_m| per rebalance month (Day2-style one-way turnover)."""
    forms = pd.DatetimeIndex(sorted(wvec.keys()))
    rows = []
    for m in range(1, len(forms)):
        d0, d1 = forms[m - 1], forms[m]
        rows.append({"decision_date": d0, "turnover": 0.5 * np.abs(wvec[d1] - wvec[d0]).sum()})
    return pd.DataFrame(rows).set_index("decision_date")["turnover"]


### 6.1 ヘッドライン実行（AUM=500億円・関与率20%・ラグ2営業日）

これが**正規のパフォーマンス指標**：自己資金化トレードポートフォリオの累積リターンです。


In [ ]:
r_trade_head, dtc_head = simulate_trade_portfolios(WVEC, AUM_TARGET, 0.20, 2)
r_head_monthly = r_trade_head.groupby(r_trade_head.index.to_period("M")).sum()
print(stats(r_head_monthly, geometric=False).round(4))
print(f"\nfinal cumulative execution P&L: {r_trade_head.cumsum().iloc[-1]*100:+.3f}%")
print(f"days_to_complete: mean {dtc_head['days_to_complete'].mean():.2f}, "
      f"median {dtc_head['days_to_complete'].median():.1f}, "
      f"share unresolved within {MAX_EXEC_DAYS}d: {dtc_head['days_to_complete'].isna().mean():.2%}")

fig, ax = plt.subplots(figsize=(9, 4.3))
ax.plot(r_trade_head.index, r_trade_head.cumsum().to_numpy() * 100, lw=1.3, color=OI[0])
ax.axhline(0, lw=0.7, color="k")
ax.set_ylabel("cumulative execution P&L (%)")
ax.set_title("Headline: self-financed trade-portfolio cumulative return\n(AUM=500oku, cap=20%, lag=2bd)")
ax.grid(alpha=0.25)
fig.tight_layout(); plt.show()


## 7. 片道ターンオーバー

In [ ]:
TURN = one_way_turnover(WVEC)
print(f"one-way turnover per rebalance: mean {TURN.mean():.2%}, median {TURN.median():.2%}, "
      f"annualised (mean x 12) {TURN.mean()*12:.2%}")
print(TURN.describe().round(4))

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(TURN.index, TURN.to_numpy() * 100, lw=1.1, color=OI[2])
ax.axhline(TURN.mean() * 100, lw=0.8, ls="--", color="0.4", label=f"mean {TURN.mean():.1%}")
ax.set_ylabel("one-way turnover (%)"); ax.set_title("Monthly one-way turnover")
ax.legend(fontsize=8); ax.grid(alpha=0.25)
fig.tight_layout(); plt.show()


## 8. メインのスイープ：関与率 × ラグ

`PARTICIPATION_GRID`（6値）× `LAG_GRID`（12値）= 72通り。939銘柄×198ヶ月規模の実データで
概算runtime 3〜5分程度（`MAX_EXEC_DAYS` を下げれば速くなります）。


In [ ]:
sweep_rows = []
curves = {}
CURVE_COMBOS = [(0.20, 2), (0.20, 20), (0.05, 2), (0.02, 10)]  # curated subset for the line chart

for cap in PARTICIPATION_GRID:
    for lag in LAG_GRID:
        r_trade, dtc = simulate_trade_portfolios(WVEC, AUM_TARGET, cap, lag)
        r_monthly = r_trade.groupby(r_trade.index.to_period("M")).sum()
        s = stats(r_monthly, geometric=False)
        sweep_rows.append({
            "participation_cap": cap, "lag_bd": lag,
            "ann_bp_yr": s["ann_%"] * 100, "vol_bp_yr": s["vol_%"] * 100, "t": s["t"],
            "cum_pct": r_trade.cumsum().iloc[-1] * 100,
            "mean_days_to_complete": dtc["days_to_complete"].mean(),
            "share_unresolved": dtc["days_to_complete"].isna().mean(),
        })
        if (cap, lag) in CURVE_COMBOS:
            curves[(cap, lag)] = r_trade.cumsum() * 100

SWEEP = pd.DataFrame(sweep_rows)
SWEEP.round(3)


In [ ]:
piv_ann = SWEEP.pivot(index="participation_cap", columns="lag_bd", values="ann_bp_yr")
piv_days = SWEEP.pivot(index="participation_cap", columns="lag_bd", values="mean_days_to_complete")
piv_unres = SWEEP.pivot(index="participation_cap", columns="lag_bd", values="share_unresolved")

fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
vmax = np.nanmax(np.abs(piv_ann.to_numpy()))
im0 = ax[0].imshow(piv_ann.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax[0].set_xticks(range(len(piv_ann.columns))); ax[0].set_xticklabels(piv_ann.columns)
ax[0].set_yticks(range(len(piv_ann.index))); ax[0].set_yticklabels([f"{c:.0%}" for c in piv_ann.index])
ax[0].set_xlabel("lag (bd)"); ax[0].set_ylabel("participation cap")
ax[0].set_title("Trade-portfolio ann. P&L (bp/yr)")
fig.colorbar(im0, ax=ax[0], pad=0.02)

im1 = ax[1].imshow(piv_days.to_numpy(), aspect="auto", cmap="viridis")
ax[1].set_xticks(range(len(piv_days.columns))); ax[1].set_xticklabels(piv_days.columns)
ax[1].set_yticks(range(len(piv_days.index))); ax[1].set_yticklabels([f"{c:.0%}" for c in piv_days.index])
ax[1].set_xlabel("lag (bd)"); ax[1].set_title("Mean days-to-complete")
fig.colorbar(im1, ax=ax[1], pad=0.02)

im2 = ax[2].imshow(piv_unres.to_numpy(), aspect="auto", cmap="magma")
ax[2].set_xticks(range(len(piv_unres.columns))); ax[2].set_xticklabels(piv_unres.columns)
ax[2].set_yticks(range(len(piv_unres.index))); ax[2].set_yticklabels([f"{c:.0%}" for c in piv_unres.index])
ax[2].set_xlabel("lag (bd)"); ax[2].set_title(f"Share unresolved within {MAX_EXEC_DAYS}d")
fig.colorbar(im2, ax=ax[2], pad=0.02)
fig.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for i, ((cap, lag), s) in enumerate(curves.items()):
    ax.plot(s.index, s.to_numpy(), lw=1.3, color=OI[i], label=f"cap={cap:.0%}, lag={lag}bd")
ax.axhline(0, lw=0.7, color="k")
ax.legend(fontsize=8); ax.set_ylabel("cumulative execution P&L (%)")
ax.set_title("Cumulative trade-portfolio return, selected (cap, lag) combinations")
ax.grid(alpha=0.25)
fig.tight_layout(); plt.show()


**読み方**：`ann_bp_yr` の符号・大きさが (関与率, ラグ) でどう変わるかが左のヒートマップ、
その裏で「そもそも執行が終わるまでどれくらいかかるか」が中央・右のヒートマップです。
`t` の絶対値がだいたい2を超えない限り、この規模のサンプルでは統計的に有意とは言えません
（Day2 §3 Part Bと同じ注意）— `SWEEP` の `t` 列も確認してください。


## 9. 流動性別の内訳と、低流動性銘柄への別処理

### 9.1 days-to-complete を流動性バケット別に


In [ ]:
r_trade_b, dtc_b = simulate_trade_portfolios(WVEC, AUM_TARGET, 0.20, 2)
dtc_b["liq_bucket"] = dtc_b["code"].map(LIQ_BUCKET)
by_bucket = dtc_b.groupby("liq_bucket")["days_to_complete"].agg(["mean", "median", "max", "count"])
print(by_bucket.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
order = ["Q1 illiquid", "Q2", "Q3", "Q4", "Q5 liquid"]
ax.bar(order, by_bucket.reindex(order)["mean"], color=OI[0], alpha=0.85)
ax.set_ylabel("mean days-to-complete"); ax.set_title("Execution speed by liquidity bucket (cap=20%, lag=2bd)")
fig.tight_layout(); plt.show()


### 9.2 低流動性銘柄（Q1）だけに別のアプローチを試す

課題の指示通り：「関与率をさらに下げる」と「そもそもの取引量（目標ウェイト差分）を絞る」を
別々に試します。**関与率を下げるのは完了を遅くするだけ**（キャパシティが減るため）で、
**目標差分を ADV の何倍までに制限するかを絞る方が、完了時間を直接コントロールできる**代わりに、
リバランスしきれない残差（トラッキングの悪化）を明示的に生みます — トレードオフの可視化です。


In [ ]:
def days_to_complete_for(delta_yen_sub, vol_window_sub, cap, aum, max_days=MAX_EXEC_DAYS):
    q = build_execution_path(delta_yen_sub, vol_window_sub, cap, aum, max_days)
    target = delta_yen_sub / aum
    reached = np.abs(q - target[None, :]) <= 1.0 / aum
    out = np.full(len(delta_yen_sub), np.nan)
    for j in range(len(delta_yen_sub)):
        if reached[:, j].any():
            out[j] = int(np.argmax(reached[:, j])) + 1
    return out

forms = pd.DatetimeIndex(sorted(WVEC.keys()))

# (a) lower participation cap, illiquid names only
rowsA = []
for cap in [0.20, 0.10, 0.05, 0.02, 0.01]:
    dtc_all = []
    for m in range(1, len(forms)):
        d0, d1 = forms[m - 1], forms[m]
        delta_w = (WVEC[d1] - WVEC[d0])[ILLIQUID_POS]
        if not (delta_w != 0).any():
            continue
        i0 = int(DPOS[shift_bd([d0], 2)[0]])
        vol_window = VOLMAT[i0:i0 + MAX_EXEC_DAYS, ILLIQUID_POS]
        dtc = days_to_complete_for(delta_w * AUM_TARGET, vol_window, cap, AUM_TARGET)
        dtc_all.append(dtc[delta_w != 0])
    dtc_all = np.concatenate(dtc_all)
    rowsA.append({"cap": cap, "mean_days": np.nanmean(dtc_all), "share_unresolved": np.isnan(dtc_all).mean()})
TABLE_A = pd.DataFrame(rowsA)
print("(a) lowering the cap for illiquid names -- slower, not faster:")
print(TABLE_A.round(3).to_string(index=False))

# (b) cap the TARGET trade size at K x ADV instead
rowsB = []
adv_illiq = ADV.to_numpy()[ILLIQUID_POS]
for k in [np.inf, 5, 2, 1, 0.5]:
    resid_all, dtc_all = [], []
    for m in range(1, len(forms)):
        d0, d1 = forms[m - 1], forms[m]
        delta_w_full = (WVEC[d1] - WVEC[d0])[ILLIQUID_POS]
        if not (delta_w_full != 0).any():
            continue
        cap_amt = k * adv_illiq / AUM_TARGET if np.isfinite(k) else np.full(len(ILLIQUID_POS), np.inf)
        delta_w_clipped = np.clip(delta_w_full, -cap_amt, cap_amt)
        residual = delta_w_full - delta_w_clipped
        i0 = int(DPOS[shift_bd([d0], 2)[0]])
        vol_window = VOLMAT[i0:i0 + MAX_EXEC_DAYS, ILLIQUID_POS]
        dtc = days_to_complete_for(delta_w_clipped * AUM_TARGET, vol_window, 0.20, AUM_TARGET)
        active = delta_w_full != 0
        dtc_all.append(dtc[active])
        resid_all.append(np.abs(residual[active]).sum())
    dtc_all = np.concatenate(dtc_all)
    rowsB.append({"size_cap_x_adv": k, "mean_days": np.nanmean(dtc_all),
                  "share_unresolved": np.isnan(dtc_all).mean(),
                  "mean_monthly_residual_weight_pp": np.mean(resid_all) * 100})
TABLE_B = pd.DataFrame(rowsB)
print("\n(b) capping the TARGET trade size at K x ADV -- bounded completion time, but leaves a residual tilt:")
print(TABLE_B.round(3).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(TABLE_A["cap"] * 100, TABLE_A["mean_days"], marker="o", color=OI[3])
ax[0].set_xlabel("participation cap (%)"); ax[0].set_ylabel("mean days-to-complete")
ax[0].set_title("(a) Lower cap -> slower (illiquid names)"); ax[0].grid(alpha=0.25)

ax2b = ax[1].twinx()
ax[1].bar(np.arange(len(TABLE_B)) - 0.15, TABLE_B["mean_days"], width=0.3, color=OI[0], label="mean days")
ax2b.bar(np.arange(len(TABLE_B)) + 0.15, TABLE_B["mean_monthly_residual_weight_pp"], width=0.3, color=OI[2], label="residual (pp)")
ax[1].set_xticks(range(len(TABLE_B))); ax[1].set_xticklabels([f"{k}x" if np.isfinite(k) else "no cap" for k in TABLE_B["size_cap_x_adv"]])
ax[1].set_ylabel("mean days-to-complete", color=OI[0]); ax2b.set_ylabel("mean residual (pp of weight)", color=OI[2])
ax[1].set_title("(b) Size-capping trades speed vs. residual tilt")
fig.tight_layout(); plt.show()


## 10. 取引コスト導入への備え（今回は実装しない、次回への申し送り）

`build_execution_path` はすでに毎日・毎銘柄の増分（`q[t]-q[t-1]`、ウェイト単位）を計算しています。
コストモデルを載せるときは、増分をその日の関与率に変換し、インパクト関数を通してから
`r_trade` を直接引く形になります：

```python
increment = q[t] - (q[t-1] if t > 0 else 0)          # weight units, signed
traded_yen = np.abs(increment) * aum
p_t = traded_yen / (vol_today + traded_yen)            # realised participation rate that day
cost_bps_t = f(p_t)                                    # e.g. linear or sqrt impact model
r_trade[t] -= (cost_bps_t / 1e4 * np.abs(increment)).sum()
```

候補：
- 単純な線形/平方根インパクト（`cost_bps = a + b*sqrt(p_t)` 等）をキャリブレーションする。
- Ohta (2026) の `ΔImp`（ラウンドプライス・クラスタリング時の追加インパクト、約+1.2bps、
  ただしsub-minuteの現象）を、日次の関与率が高い日に対する**上振れ調整**として部分的に
  組み込む — ただし sub-minute → daily の橋渡しは拡張が大きいので要相談。
- 関与率スイープ（§8）の結果と合わせれば、「コストを載せた後、どの (関与率, ラグ) が
  ネットで一番良いか」という最終的な問いに直結します。


## 11. 設計判断メモ

1. **フレームワークを「全体NAVの日次ドリフト」から「月ごとの自己資金化トレードポートフォリオを
   独立に足し合わせる」方式に変更**。理由は頑健性（§0参照）と、指示された規約とも自然に整合する
   ため。副作用として、実行が翌月に食い込んでも「バックログ」を無理に翌月の差分に混ぜ込む必要が
   なくなった（重複月はそのまま足し合わせるだけ）。
2. **累積は算術（単純合計）**。Day2 の `stats()` 自身の注記（差分系列を複利で積むのは分散ドラッグを
   コストに見せかけるだけ）に合わせている。
3. **AUMは名目値**（複利で増減しない）として関与率の計算に使用— 変える場合は Setup の
   `AUM_TARGET` を書き換えるだけ。
4. **リターンの健全性チェックを追加**（§1）。実データで見つかった `-243%` の maxdd はほぼ確実に
   データ側の異常値が原因。クリップ後の結果と、クリップ前後でどれだけ変わるかは一度確認することを
   お勧めします。
5. `lag` は「月末決定日から執行開始までの営業日数」（以前の `start_lag` と同じ意味）。
